In [2]:
# Core imports
import numpy as np
import torch
import sys
from pathlib import Path
# Add project root to Python path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Import modules
from src.data.dataset import create_data_loaders
from src.models.deeplog import DeepLogModel
from src.engine.trainer import LogSeqTrainer
from src.utils.metrics import evaluate_model, print_metrics, save_experiment_results
from src.utils.data_loader import create_train_val_test_split, filter_normal_samples, load_loghub
from src.utils.visualizer import UniversalAnomalyVisualizer
from src.utils.seed import seed_everything

In [3]:
seed_everything(42)  # For reproducibility

In [4]:
# Define paths
DATA_DIR = '../data/bgl/preprocessed'

In [5]:
X, y, vocab = load_loghub(DATA_DIR)
vocab_size = len(vocab)
print(f"Vocab size: {vocab_size}")
print(f"Events: {sorted(vocab.keys())[:10]}")  # See first 10

INFO:src.utils.data_loader:Loading data from LogHub preprocessing: ../data/bgl/preprocessed
INFO:src.utils.data_loader:Loaded data:
INFO:src.utils.data_loader:  - Sequences: (949589, 20)
INFO:src.utils.data_loader:  - Labels: (949589,)
INFO:src.utils.data_loader:  - Normal: 868556, Anomaly: 81033
INFO:src.utils.data_loader:  - Loaded vocabulary: 371 events


Vocab size: 371
Events: ['<PAD>', '<UNK>', 'E1', 'E10', 'E100', 'E101', 'E102', 'E103', 'E1035', 'E1036']


In [6]:
#  Split data (70/15/15)
splits = create_train_val_test_split(X, y, train_ratio=0.7, val_ratio=0.15, random_state=42)
(X_train, y_train), (X_val, y_val), (X_test, y_test) = splits['train'], splits['val'], splits['test']
X_train, y_train = filter_normal_samples(X_train, y_train, verbose=True)

INFO:src.utils.data_loader:Splitting data: train=0.7, val=0.15, test=0.15
INFO:src.utils.data_loader:Split complete:
INFO:src.utils.data_loader:  - Train: 664711 samples (56723 anomalies)
INFO:src.utils.data_loader:  - Val:   142439 samples (12155 anomalies)
INFO:src.utils.data_loader:  - Test:  142439 samples (12155 anomalies)
INFO:src.utils.data_loader:======================================================================
INFO:src.utils.data_loader:FILTERING TRAINING DATA FOR SEMI-SUPERVISED LEARNING
INFO:src.utils.data_loader:======================================================================
INFO:src.utils.data_loader:Original training size: 664711 samples
INFO:src.utils.data_loader:  Normal samples: 607,988 (91.47%)
INFO:src.utils.data_loader:  Anomaly samples: 56,723 (8.53%)
INFO:src.utils.data_loader:
Filtered training size: 607,988 samples (NORMAL ONLY)
INFO:src.utils.data_loader:Removed 56,723 anomalies from training set
INFO:src.utils.data_loader:✓ Training data is now pur

In [7]:
print(f"Unique events in first 100 train sequences: {len(set(e for seq in X_train[:100] for e in seq))}")
print(f"Max event ID: {max(e for seq in X_train for e in seq)}")
print(f"Min event ID: {min(e for seq in X_train for e in seq)}")
print(f"Train shape: {X_train.shape}")

Unique events in first 100 train sequences: 40
Max event ID: 370
Min event ID: 1
Train shape: (607988, 20)


In [8]:
batch_size = 64
train_loader, val_loader, test_loader = create_data_loaders(
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    batch_size=batch_size
)

In [9]:
#  Create model
model = DeepLogModel(
    vocab_size=vocab_size,
    embedding_dim=64,
    hidden_dim=128,
    num_layers=2,
    dropout=0.3 
)

In [10]:
# Train
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
learning_rate = 0.001
trainer = LogSeqTrainer(model, device=device, learning_rate=learning_rate)
patience = 20
history = trainer.fit(
    train_loader, val_loader,
    num_epochs=50,
    early_stopping_patience=patience,
    print_every=5
)

Training: 100%|██████████| 9500/9500 [00:43<00:00, 218.95it/s]



Epoch 1/50 - 55.88s
  Train Loss: 0.2196
  Val Loss:   0.9349
  ✓ New best model (val_loss: 0.9349)


Training: 100%|██████████| 9500/9500 [00:42<00:00, 222.77it/s]



Epoch 5/50 - 55.19s
  Train Loss: 0.1661
  Val Loss:   0.7219
  ✓ New best model (val_loss: 0.7219)


Training: 100%|██████████| 9500/9500 [00:42<00:00, 221.18it/s]


...


Epoch 35/50 - 55.22s
  Train Loss: 0.1630
  Val Loss:   0.7242
  No improvement (12/20)


Training: 100%|██████████| 9500/9500 [00:42<00:00, 222.60it/s]



Epoch 40/50 - 55.29s
  Train Loss: 0.1630
  Val Loss:   0.7211
  No improvement (17/20)


Training: 100%|██████████| 9500/9500 [00:42<00:00, 224.15it/s]



Early stopping triggered after 43 epochs

✓ Loaded best model (val_loss: 0.6912)
Total training time: 2376.57s


In [11]:
# Evaluate
# Capture predictions, true labels, AND anomaly scores from the loader
top_k = 11
predictions, true_labels, anomaly_scores = trainer.detect_anomalies(test_loader, top_k=top_k, return_scores=True)
metrics = evaluate_model(predictions, true_labels) 
print_metrics(metrics)

Detecting anomalies: 100%|██████████| 2226/2226 [00:21<00:00, 105.28it/s]


EVALUATION METRICS
Accuracy:  0.9883 (98.83%)
Precision: 0.8840
Recall:    0.9935
F1-Score:  0.9356

Confusion Matrix:
              Predicted
              Normal  Anomaly
Actual Normal   128700     1584
       Anomaly      79    12076


In [14]:
max_len = int(np.percentile([len(s) for s in X_train], 95))

In [13]:
#  Save results
save_experiment_results(
    filepath="../results/bgl_deeplog_results.json",
    dataset="BGL",
    model_name="DeepLog",
    device=device,
    model=model,
    history=history,
    y_train=y_train,
    y_val=y_val,
    y_test=y_test,
    max_len=max_len,
    metrics=metrics,
    batch_size=batch_size,
    learning_rate=learning_rate,
    patience=patience,
    top_k=top_k
)

✓ Results saved to ../results/bgl_deeplog_results.json


{'dataset': 'BGL',
 'model': 'DeepLog',
 'device': 'mps',
 'architecture': {'vocab_size': 371,
  'embedding_dim': 64,
  'hidden_dim': 128,
  'num_layers': 2,
  'dropout': 0.3},
 'training': {'num_epochs': 43,
  'batch_size': 64,
  'learning_rate': 0.001,
  'early_stopping_patience': 20,
  'best_val_loss': 0.6911504271190014,
  'training_time_seconds': 2376.573264837265},
 'data': {'train_size': 607988,
  'val_size': 142439,
  'test_size': 142439,
  'max_sequence_length': 20,
  'train_normal_only': True},
 'detection': {'top_k': 11, 'method': 'next_event_prediction'},
 'metrics': {'accuracy': 0.9883248267679497,
  'precision': 0.8840409956076135,
  'recall': 0.9935006170300288,
  'f1': 0.9355800890954871,
  'confusion_matrix': [[128700, 1584], [79, 12076]]}}